In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

PROJECT_FOLDER = "vehicle_parts_project"      # the folder name in your Google Drive (Colab only)

def find_root():
    # 1) running as a file inside the project (python run_all.py): the folder above notebooks/
    try:
        here = Path(__file__).resolve().parent
        if here.name == "notebooks":
            return here.parent
    except NameError:
        pass
    # 2) Google Colab: MyDrive/<PROJECT_FOLDER> (asks for Drive permission once)
    drive_root = Path("/content/drive/MyDrive")
    try:
        from google.colab import drive
        if not drive_root.exists():
            drive.mount("/content/drive")
    except Exception:
        pass
    if drive_root.exists():
        return drive_root / PROJECT_FOLDER
    # 3) anywhere else: the current folder (or its parent if you are inside notebooks/)
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd

ROOT = find_root()

PROC, FIG = ROOT / "data" / "processed", ROOT / "figure"
FIG.mkdir(exist_ok=True)

# ---- STARTING WEIGHTS: the brief's weights are "example only". Change them and justify the choice in the report.
WEIGHTS = {"demand": 0.30, "scarcity": 0.25, "compatibility": 0.20, "sourcing": 0.15, "confidence": 0.10}
assert abs(sum(WEIGHTS.values()) - 1) < 1e-9, "weights must add up to 1"
TOP_N = 5

# ---- ANALYST JUDGEMENT: how much does each fitment status count? Justify these numbers in the report.
COMPAT_SCORE = {"confirmed_2_sources": 1.0, "listed_in_2_sources_codes_differ": 0.8,
                "confirmed_1_source": 0.6, "possible_fit": 0.3, "not_found": 0.0}

Mounted at /content/drive


## 1. Factors and score

In [ ]:
df = pd.read_csv(PROC / "cleaned_dataset.csv")

def minmax(s):
    lo, hi = s.min(), s.max()
    if pd.isna(lo) or hi == lo:
        return s.where(s.isna(), 0.5)      # no spread -> neutral 0.5, NaN stays NaN
    return (s - lo) / (hi - lo)

if df["price_usd"].notna().any():
    price, price_unit = df["price_usd"], "USD"
else:
    units = set(df.loc[df["price_original"].notna(), "currency_original"].dropna())
    assert len(units) <= 1, f"prices in several currencies {units} - set an FX rate and re-run 01_clean_data"
    price, price_unit = df["price_original"], (units.pop() if units else "none")
print("Price column used for scoring:", price_unit)

f = df[["vehicle", "part"]].copy()
f["demand"] = pd.concat([df["wear_item"], minmax(df["nhtsa_complaints"])], axis=1).mean(axis=1, skipna=True)
f["scarcity"] = 1 - minmax(df["local_sellers"])
f["compatibility"] = df["compat_status"].map(COMPAT_SCORE)
f["sourcing"] = pd.concat([df["in_stock"], 1 - minmax(price)], axis=1).mean(axis=1, skipna=True)
key = ["oem_number", "price_original", "in_stock", "wear_item", "nhtsa_complaints", "local_sellers"]
found = df[key].notna().sum(axis=1)
f["confidence"] = 0.5 * (found / len(key)) + 0.5 * (df["n_sources"].fillna(0).clip(upper=2) / 2)

FACTORS = list(WEIGHTS)
ACTIVE = [k for k in FACTORS if f[k].notna().any()]
DROPPED = [k for k in FACTORS if k not in ACTIVE]
if DROPPED:
    print("NOTE: no data at all for", DROPPED, "- left out and the other weights re-scaled. Say this in the report.")
used_w = {k: WEIGHTS[k] / sum(WEIGHTS[a] for a in ACTIVE) for k in ACTIVE}
print("Weights actually used:", {k: round(v, 3) for k, v in used_w.items()})

def score(frame, weights):
    tot = sum(weights[k] for k in ACTIVE)
    return sum(weights[k] / tot * frame[k].fillna(0) for k in ACTIVE)

f["score"] = score(f, WEIGHTS)
f.round(3).to_csv(PROC / "scored_vehicle_part.csv", index=False)
f.round(3).sort_values("score", ascending=False).head(10)

Price column used for scoring: RMB
NOTE: no data at all for ['scarcity'] - left out and the other weights re-scaled. Say this in the report.
Weights actually used: {'demand': 0.4, 'compatibility': 0.267, 'sourcing': 0.2, 'confidence': 0.133}


,vehicle,part,demand,scarcity,compatibility,sourcing,confidence,score
1,BYD Seagull,Brake pads,NaN,NaN,1.0,1.000,0.667,0.556
0,BYD Seagull,Brake discs,NaN,NaN,1.0,0.961,0.667,0.548
6,BYD Seagull,Shock absorbers,NaN,NaN,1.0,0.922,0.667,0.540
2,BYD Seagull,Bumpers,NaN,NaN,1.0,0.876,0.667,0.531
8,BYD Seagull,Tail lights,NaN,NaN,1.0,0.859,0.667,0.527
7,BYD Seagull,Side mirrors,NaN,NaN,1.0,0.702,0.667,0.496
5,BYD Seagull,Headlights,NaN,NaN,1.0,0.627,0.667,0.481
24,Volkswagen ID.4/ID.6,Fenders/body panels,1.000,NaN,0.0,NaN,0.333,0.444
20,Volkswagen ID.4/ID.6,Brake discs,0.825,NaN,0.0,NaN,0.333,0.374
21,Volkswagen ID.4/ID.6,Brake pads,0.825,NaN,0.0,NaN,0.333,0.374


## 2. Part ranking (average over the three vehicles)

In [ ]:
part = f.assign(**{k: f[k].fillna(0) for k in ACTIVE}).groupby("part")[ACTIVE + ["score"]].mean()   # missing counts as 0, so factor columns add up to the score
ev = df.assign(has_fit=df["compat_status"].ne("not_found")).groupby("part")["has_fit"].sum().rename("vehicles_with_fitment_evidence")
part = part.join(ev)
part["coverage_pct"] = (100 * df.groupby("part")["n_missing_key"].apply(lambda s: 1 - s.mean() / len(key))).round(0)
part = part.sort_values(["score", "vehicles_with_fitment_evidence"], ascending=False)
part["rank"] = range(1, len(part) + 1)
part.round(3).to_csv(PROC / "part_ranking.csv")
if part["score"].nunique() == 1:
    print("WARNING: every part has the same score - there is almost no evidence in the sheet.")
    print("         This ranking is NOT meaningful. Collect and fill in the evidence first (see README).")
top5 = part.head(TOP_N)
top5.round(3).to_csv(PROC / "top5_parts.csv")

gap = part["score"].iloc[TOP_N - 1] - part["score"].iloc[TOP_N]
print(f"Gap between rank {TOP_N} and rank {TOP_N + 1}: {gap:.3f}  (a small gap means the cut-off is fragile)")
json.dump({"weights_set": WEIGHTS, "weights_used": used_w, "factors_dropped": DROPPED,
           "compat_score": COMPAT_SCORE, "price_unit": price_unit}, open(PROC / "scoring_config.json", "w"), indent=2)
top5.round(3)

Gap between rank 5 and rank 6: 0.008  (a small gap means the cut-off is fragile)


,demand,compatibility,sourcing,confidence,score,vehicles_with_fitment_evidence,coverage_pct,rank
part,,,,,,,,
Brake pads,0.275,0.333,0.333,0.333,0.310,1,17.0,1
Brake discs,0.275,0.333,0.320,0.333,0.307,1,17.0,2
Fenders/body panels,0.333,0.333,0.000,0.333,0.267,1,17.0,3
Tail lights,0.097,0.333,0.286,0.333,0.229,1,17.0,4
Side mirrors,0.106,0.333,0.234,0.333,0.222,1,17.0,5


## 3. Does changing the weights change the ranking?

In [ ]:
scenarios = {
    "Base": WEIGHTS,
    "Equal weights": {k: 0.2 for k in FACTORS},
    "Demand-heavy": {"demand": 0.50, "scarcity": 0.15, "compatibility": 0.15, "sourcing": 0.10, "confidence": 0.10},
    "Scarcity-heavy": {"demand": 0.20, "scarcity": 0.45, "compatibility": 0.15, "sourcing": 0.10, "confidence": 0.10},
    "Sourcing-heavy": {"demand": 0.20, "scarcity": 0.15, "compatibility": 0.15, "sourcing": 0.40, "confidence": 0.10},
    "Compatibility-heavy": {"demand": 0.20, "scarcity": 0.15, "compatibility": 0.45, "sourcing": 0.10, "confidence": 0.10},
}
ranks = {}
for name, w in scenarios.items():
    assert abs(sum(w.values()) - 1) < 1e-9, name
    s = f.assign(s=score(f, w)).groupby("part")["s"].mean()
    ranks[name] = s.rank(ascending=False, method="min").astype(int)
ranks = pd.DataFrame(ranks).loc[part.index]
ranks.to_csv(PROC / "sensitivity_ranks.csv")

base_top = set(ranks.index[ranks["Base"] <= TOP_N])
print("Parts shared with the base top 5, by scenario:",
      {c: len(base_top & set(ranks.index[ranks[c] <= TOP_N])) for c in ranks.columns})

rng = np.random.default_rng(42)
hits = pd.Series(0.0, index=part.index)
N_RUNS = 1000
for w in rng.dirichlet(np.ones(len(FACTORS)), size=N_RUNS):
    s = f.assign(s=score(f, dict(zip(FACTORS, w)))).groupby("part")["s"].mean()
    hits[s.nlargest(TOP_N).index] += 1
stability = (100 * hits / N_RUNS).rename("pct_runs_in_top5").round(1)
stability.to_csv(PROC / "sensitivity_stability.csv")
pd.concat([ranks, stability], axis=1)

Parts shared with the base top 5, by scenario: {'Base': 5, 'Equal weights': 5, 'Demand-heavy': 5, 'Scarcity-heavy': 5, 'Sourcing-heavy': 3, 'Compatibility-heavy': 5}


,Base,Equal weights,Demand-heavy,Scarcity-heavy,Sourcing-heavy,Compatibility-heavy,pct_runs_in_top5
part,,,,,,,
Brake pads,1,1,1,1,1,1,100.0
Brake discs,2,2,2,2,2,2,100.0
Fenders/body panels,3,5,3,3,8,3,51.3
Tail lights,4,3,4,4,3,4,99.3
Side mirrors,5,4,5,5,6,5,65.2
Headlights,6,7,6,6,7,6,0.0
Shock absorbers,7,6,8,7,4,7,48.7
Bumpers,8,8,9,8,5,8,34.8
Wiper blades,9,9,7,9,9,9,0.7


## 4. Figures

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
order = part.index[::-1]
colors = ["#1f6f5c" if part.loc[p, "rank"] <= TOP_N else "#b8c4c2" for p in order]

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(order, part.loc[order, "score"], color=colors)
ax.set_xlabel("Priority score (0 to 1)"); ax.set_title("Parts ranked by import priority (green = recommended)")
for i, p in enumerate(order):
    ax.text(part.loc[p, "score"] + 0.005, i, f"{part.loc[p, 'score']:.2f}", va="center")
fig.tight_layout(); fig.savefig(FIG / "01_part_ranking.png", dpi=200); plt.close(fig)

contrib = pd.DataFrame({k: part[k] * used_w[k] for k in ACTIVE}).loc[order]
fig, ax = plt.subplots(figsize=(8, 4.8))
left = np.zeros(len(contrib))
for k, c in zip(ACTIVE, ["#1f6f5c", "#e0a030", "#3b6ea5", "#a5483b", "#8a8a8a"]):
    ax.barh(contrib.index, contrib[k], left=left, label=k.capitalize(), color=c); left += contrib[k].values
ax.set_xlabel("Weighted score contribution"); ax.set_title("Where each part's score comes from")
ax.legend(frameon=False, ncol=3, loc="lower right", fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "02_score_breakdown.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.imshow(ranks.values, cmap="Greens_r", aspect="auto", vmin=1, vmax=len(ranks))
ax.set_xticks(range(ranks.shape[1])); ax.set_xticklabels(ranks.columns, rotation=30, ha="right")
ax.set_yticks(range(ranks.shape[0])); ax.set_yticklabels(ranks.index)
for i in range(ranks.shape[0]):
    for j in range(ranks.shape[1]):
        ax.text(j, i, ranks.values[i, j], ha="center", va="center", fontsize=9, color="white" if ranks.values[i, j] <= 3 else "black")
ax.set_title("Rank of each part under different weights (1 = highest)")
fig.tight_layout(); fig.savefig(FIG / "03_weight_sensitivity.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.8))
st = stability.loc[order]
ax.barh(st.index, st.values, color=colors)
ax.set_xlabel("% of 1,000 random weight sets in which the part is in the top 5"); ax.set_xlim(0, 100)
ax.set_title("How stable is each recommendation?")
fig.tight_layout(); fig.savefig(FIG / "04_ranking_stability.png", dpi=200); plt.close(fig)
print("Saved 4 figures to", FIG)

Saved 4 figures to /content/drive/MyDrive/vehicle_parts_project/figure
